# Notebook: Pipeline de ML simples

Este caderno mostra um fluxo básico de preparação, treinamento e avaliação de um modelo de classificação.

In [ ]:
# Importar bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay

plt.style.use('seaborn-v0_8')

In [ ]:
# Configurar dados de exemplo

data = {
    "idade": [25, 32, 47, 51, 62, 23, 34, 45, 52, 29, 38, 41, 57, 49, 36],
    "renda": [3000, 4500, 7000, 8000, 12000, 2800, 4600, 6800, 9000, 3200, 5200, 6100, 11000, 8400, 5600],
    "cidade": ["RJ", "SP", "SP", "MG", "RS", "RJ", "SP", "MG", "RS", "RJ", "SP", "MG", "RS", "SP", "RJ"],
    "compra": [0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 0],
}

df = pd.DataFrame(data)
df.head()

In [ ]:
# Pré-processar dados

X = df.drop(columns=["compra"])
y = df["compra"]

cat_cols = ["cidade"]
num_cols = [c for c in X.columns if c not in cat_cols]

categorical_transformer = OneHotEncoder(handle_unknown="ignore")
numeric_transformer = Pipeline(steps=[("scaler", StandardScaler())])

preprocess = ColumnTransformer(
    transformers=[
        ("cat", categorical_transformer, cat_cols),
        ("num", numeric_transformer, num_cols),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, X_test.shape

In [ ]:
# Treinar modelo

clf = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("model", RandomForestClassifier(n_estimators=150, random_state=42)),
    ]
)

clf.fit(X_train, y_train)

train_pred = clf.predict(X_train)
test_pred = clf.predict(X_test)

train_acc = accuracy_score(y_train, train_pred)
test_acc = accuracy_score(y_test, test_pred)

train_acc, test_acc

In [ ]:
# Avaliar e exibir métricas

cm = confusion_matrix(y_test, test_pred)
print(f"Acurácia treino: {train_acc:.3f} | teste: {test_acc:.3f}")

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(cm).plot(ax=ax, values_format='d')
plt.show()

In [ ]:
# Salvar artefatos

import joblib
from pathlib import Path

artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

model_path = artifacts_dir / "rf_model.joblib"
metrics_path = artifacts_dir / "metrics.json"

joblib.dump(clf, model_path)

import json
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({"train_acc": train_acc, "test_acc": test_acc}, f, ensure_ascii=False, indent=2)

model_path, metrics_path